# Question Bank Preparation

Builds a filtered, domain-balanced question bank for C4 ablation sweep.

**Criteria for inclusion:**
1. Pythia 410M and Pythia 1.4B **disagree at first token** (different Phase B predictions)
2. **Robust disagreement**: `P_410M(own_answer) > 2 × P_410M(other_answer)` — 410M strongly prefers its own answer over 1.4B's
3. **Domain balance**: final bank has ≥ 5 questions from each of 6 categories

**Pipeline:**
- ~100 candidate questions across 6 domains
- Run Phase B on both models, extract first-token logits
- Apply filters
- Domain-balance final selection to 40 questions
- Save as `question_bank.json` with metadata (domain tag, 410M answer, 1.4B answer, disagreement strength)

**Runtime:** ~10-15 minutes on T4.

In [ ]:
!pip install -q transformers datasets torch accelerate

In [ ]:
import torch, json, os, time
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

OUT_DIR = './preflight'
try:
    from google.colab import drive
    try:
        drive.mount('/content/drive')
        OUT_DIR = '/content/drive/MyDrive/DFE_research/preflight'
    except Exception as e:
        print(f'Drive failed ({e}); using local')
except ImportError:
    pass
os.makedirs(OUT_DIR, exist_ok=True)

def log(msg):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True)

In [ ]:
log('Loading Pythia 410M + 1.4B (both at step 143000)...')
tok = AutoTokenizer.from_pretrained('EleutherAI/pythia-410m-deduped')  # shared tokenizer

m410 = AutoModelForCausalLM.from_pretrained(
    'EleutherAI/pythia-410m-deduped', revision='step143000', torch_dtype=torch.float16
).to(device).eval()
m14b = AutoModelForCausalLM.from_pretrained(
    'EleutherAI/pythia-1.4b-deduped', revision='step143000', torch_dtype=torch.float16
).to(device).eval()

if device == 'cuda':
    print(f'GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB')

## Candidate questions (100 across 6 domains)

In [ ]:
CANDIDATES = {
    'geography': [
        'What is the capital of France?', 'What is the capital of Japan?', 'What is the capital of Italy?',
        'What is the capital of Germany?', 'What is the capital of Spain?', 'What is the capital of Russia?',
        'What is the capital of Canada?', 'What is the capital of Egypt?', 'What is the capital of Argentina?',
        'What is the largest country in the world?', 'What is the longest river in the world?',
        'What is the tallest mountain on Earth?', 'In which ocean is Hawaii located?',
        'Which continent is Egypt in?', 'Which country has the largest population?',
        'What is the largest desert in the world?', 'What river flows through Paris?',
        'What mountain range separates Europe from Asia?', 'What is the smallest continent?',
        'In which country is the Great Wall located?',
    ],
    'history': [
        'What year did World War Two end?', 'Who was the first president of the United States?',
        'In what year did the Titanic sink?', 'Who wrote the Declaration of Independence?',
        'What empire built the Colosseum?', 'Who was the first emperor of Rome?',
        'In what year did Columbus reach America?', 'What year did the Berlin Wall fall?',
        'Who was the leader of Nazi Germany?', 'What year was the French Revolution?',
        'Who led the Soviet Union during World War Two?', 'What year did World War One start?',
        'Who was the first woman prime minister of the UK?', 'What year did the American Civil War begin?',
        'Who was the first pharaoh of united Egypt?',
    ],
    'science': [
        'What gas do humans breathe in?', 'What is the chemical formula for water?',
        'What is the speed of light in a vacuum?', 'What planet is known as the Red Planet?',
        'What is the largest planet in our solar system?', "What gas makes up most of Earth's atmosphere?",
        'What is the hardest natural substance on Earth?', 'What is the smallest unit of matter?',
        'What is the chemical symbol for gold?', 'What is the boiling point of water in Celsius?',
        'What force keeps planets in orbit?', 'What organ in the human body pumps blood?',
        'How many bones are in the adult human body?', 'What is the most abundant gas in the sun?',
        'What is the closest star to Earth?', 'What are the three states of matter?',
        'What is the main source of energy for plants?', 'What is the pH of pure water?',
    ],
    'literature': [
        'Who wrote the play Romeo and Juliet?', 'Who wrote the novel Pride and Prejudice?',
        'Who is the author of Hamlet?', 'Who painted the Mona Lisa?', 'Who composed the Ninth Symphony?',
        'Who wrote War and Peace?', 'Who wrote Don Quixote?', 'Who wrote the Divine Comedy?',
        'Who wrote The Great Gatsby?', 'Who wrote Crime and Punishment?',
        'Who painted The Starry Night?', 'Who wrote 1984?', 'Who composed The Four Seasons?',
        'Who sculpted the David?', 'Who wrote To Kill a Mockingbird?',
    ],
    'math': [
        'What is two plus two?', 'What is the square root of 144?', 'How many sides does a hexagon have?',
        'What is the smallest prime number?', 'How many degrees are in a right angle?',
        'What is ten times ten?', 'How many sides does a pentagon have?', 'What is pi rounded to two decimals?',
        'How many zeros in one million?', 'What is five squared?',
        'How many degrees in a full circle?', 'What is half of one hundred?',
        'How many minutes are in an hour?', 'How many hours are in a day?',
    ],
    'common_sense': [
        'What color is the sky on a clear day?', 'What do bees collect from flowers?',
        'What is the main ingredient in bread?', 'What language is spoken in Brazil?',
        'How many continents are there?', 'What is the currency of the United Kingdom?',
        'What do cows produce?', 'What is the opposite of hot?',
        'What color do you get when you mix red and blue?', 'What vehicle travels in the ocean?',
        'What do you use to cut paper?', 'What falls from clouds as rain?',
        'What season comes after summer?', 'What do birds use to fly?',
    ],
}

flat_questions = []
for dom, qs in CANDIDATES.items():
    for q in qs:
        flat_questions.append({'q': q, 'domain': dom})
print(f'Total candidates: {len(flat_questions)} across {len(CANDIDATES)} domains')
for dom, qs in CANDIDATES.items():
    print(f'  {dom}: {len(qs)}')

## Phase B runner with full-distribution capture

In [ ]:
PHASE_B = '''Question: What is 5 + 3? Answer: Eight.
Question: Who wrote Hamlet? Answer: Shakespeare.
Question: What color is grass? Answer: Green.
Question: {q} Answer:'''

@torch.no_grad()
def first_token_probs(model, tok, prompt):
    """Return (top_token_str, full_logprob_vector, softmax_probs).
    We need the full distribution to compute P(any given token)."""
    inputs = tok(prompt, return_tensors='pt', truncation=True, max_length=1024).to(device)
    logits = model(**inputs).logits[0, -1, :]
    probs = torch.softmax(logits, dim=-1)
    top_id = logits.argmax().item()
    return tok.decode([top_id]).strip().lower(), probs.cpu().numpy()

def prob_of_first_token_of(text, probs):
    """Given a full prob vector and a text answer, return P(first token of text)."""
    if not text.strip():
        return 0.0
    # Try both with-space and without-space variants
    best = 0.0
    for variant in [' ' + text, text]:
        ids = tok.encode(variant, add_special_tokens=False)
        if ids:
            p = float(probs[ids[0]])
            if p > best: best = p
    return best

# Quick sanity check
t410, p410 = first_token_probs(m410, tok, PHASE_B.format(q='What is the capital of France?'))
t14, p14 = first_token_probs(m14b, tok, PHASE_B.format(q='What is the capital of France?'))
print(f'410M: top_token="{t410}", P(paris)={prob_of_first_token_of("paris", p410):.3f}')
print(f'1.4B: top_token="{t14}", P(paris)={prob_of_first_token_of("paris", p14):.3f}')

## Run Phase B on all 100 candidates

In [ ]:
log('Running Phase B on candidates...')
results = []
for i, item in enumerate(flat_questions):
    q = item['q']
    prompt = PHASE_B.format(q=q)
    t410, p410 = first_token_probs(m410, tok, prompt)
    t14, p14 = first_token_probs(m14b, tok, prompt)
    
    # Cross-probabilities for robustness filter
    p410_own = prob_of_first_token_of(t410, p410)
    p410_other = prob_of_first_token_of(t14, p410)
    p14_own = prob_of_first_token_of(t14, p14)
    p14_other = prob_of_first_token_of(t410, p14)
    
    results.append({
        'q': q,
        'domain': item['domain'],
        'answer_410m': t410,
        'answer_14b': t14,
        'disagree': t410 != t14,
        'p410_own': p410_own,
        'p410_other': p410_other,
        'p14_own': p14_own,
        'p14_other': p14_other,
        'robust_410': p410_own > 2 * p410_other if p410_other > 0 else p410_own > 0.1,
        'robust_14': p14_own > 2 * p14_other if p14_other > 0 else p14_own > 0.1,
    })
    if (i+1) % 20 == 0:
        log(f'  [{i+1}/{len(flat_questions)}]')

log('Done.')
n_disagree = sum(1 for r in results if r['disagree'])
n_robust = sum(1 for r in results if r['disagree'] and r['robust_410'] and r['robust_14'])
print(f'\nDisagreement: {n_disagree}/{len(results)}')
print(f'Robust disagreement (both models strongly prefer own): {n_robust}/{len(results)}')

## Apply filters: robust disagreement + domain balance

In [ ]:
# Step 1: filter to robust disagreements
robust = [r for r in results if r['disagree'] and r['robust_410'] and r['robust_14']]
print(f'After robust filter: {len(robust)} questions')
print('\nBy domain:')
from collections import Counter
domain_counts = Counter(r['domain'] for r in robust)
for dom, cnt in domain_counts.items():
    print(f'  {dom}: {cnt}')

# Step 2: domain-balance — take up to N_PER_DOMAIN from each
TARGET_TOTAL = 40
n_domains_with_items = sum(1 for cnt in domain_counts.values() if cnt > 0)
N_PER_DOMAIN = max(TARGET_TOTAL // max(n_domains_with_items, 1), 3)

# Rank within each domain by 'disagreement strength' = min(P_own / P_other) for both models
def disagreement_strength(r):
    ratio_410 = r['p410_own'] / max(r['p410_other'], 1e-6)
    ratio_14 = r['p14_own'] / max(r['p14_other'], 1e-6)
    return min(ratio_410, ratio_14)  # both must be strong

for r in robust:
    r['strength'] = disagreement_strength(r)

by_domain = {}
for r in robust:
    by_domain.setdefault(r['domain'], []).append(r)
for dom in by_domain:
    by_domain[dom].sort(key=lambda r: r['strength'], reverse=True)

# Selection: take top N_PER_DOMAIN from each domain, fill up to TARGET_TOTAL
selected = []
for dom, items in by_domain.items():
    selected.extend(items[:N_PER_DOMAIN])
# If we still need more, take the next-best from all remaining by strength
if len(selected) < TARGET_TOTAL:
    remaining = [r for r in robust if r not in selected]
    remaining.sort(key=lambda r: r['strength'], reverse=True)
    selected.extend(remaining[:TARGET_TOTAL - len(selected)])
# If too many, trim by lowest strength
if len(selected) > TARGET_TOTAL:
    selected.sort(key=lambda r: r['strength'], reverse=True)
    selected = selected[:TARGET_TOTAL]

print(f'\nFinal selection: {len(selected)} questions')
print('By domain:')
for dom, cnt in Counter(r['domain'] for r in selected).items():
    print(f'  {dom}: {cnt}')

## Inspection: final question bank

In [ ]:
print(f'{"domain":<14} | {"question":<45} | {"410M":<13} {"1.4B":<13} | strength')
print('=' * 120)
for r in sorted(selected, key=lambda x: (x['domain'], -x['strength'])):
    print(f"{r['domain']:<14} | {r['q'][:43]:<45} | {r['answer_410m']:<13} {r['answer_14b']:<13} | {r['strength']:.1f}")

## Save question bank artifact

In [ ]:
# Save full result dataset (includes all candidates, filter audit trail)
with open(os.path.join(OUT_DIR, 'question_bank_candidates_full.json'), 'w') as f:
    json.dump({
        'n_candidates': len(results),
        'n_disagree': n_disagree,
        'n_robust': n_robust,
        'results': results,
    }, f, indent=2)

# Save filtered question bank (primary artifact for micro-pilot)
bank = {
    'n_questions': len(selected),
    'filter_criteria': {
        'robust_threshold': '2x: P_model(own) > 2 * P_model(other)',
        'models': ['Pythia 410M step143000', 'Pythia 1.4B step143000'],
        'phase_b_template': PHASE_B,
    },
    'questions': [
        {
            'q': r['q'],
            'domain': r['domain'],
            'answer_410m': r['answer_410m'],
            'answer_14b': r['answer_14b'],
            'p410_own': r['p410_own'],
            'p410_other': r['p410_other'],
            'p14_own': r['p14_own'],
            'p14_other': r['p14_other'],
            'strength': r['strength'],
        }
        for r in selected
    ],
}
with open(os.path.join(OUT_DIR, 'question_bank.json'), 'w') as f:
    json.dump(bank, f, indent=2)

print(f'Saved:')
print(f'  {OUT_DIR}/question_bank_candidates_full.json (audit trail, {len(results)} candidates)')
print(f'  {OUT_DIR}/question_bank.json (final bank, {len(selected)} questions)')
print('\nDownload question_bank.json to commit to GitHub repo.')